In [1]:
import os
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [157]:
from sparkSession import spark
from pyspark.sql.functions import *
from pyspark.sql.types import DatetimeType
from pyspark.sql.types import *
from pyspark.sql import Window

spark.sparkContext.setLogLevel("ERROR")

In [15]:
READ_PATH_CLICKSTREAM = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_clickstream'
READ_PATH_USER = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_user'

In [194]:
dataframe_clickstream = spark.read.format('delta').load(READ_PATH_CLICKSTREAM)
dataframe_user = spark.read.format('delta').load(READ_PATH_USER)

In [17]:
# uses dictionary comprehension
def trim_everything_lower(dataframe):
    df_trimmed = dataframe.withColumns({c: lower(trim(col(c))) for c in dataframe.columns})
    return df_trimmed

In [35]:
## ensures unique value on passed field list
def ensure_unique(dataframe, *fields):
    df_unique = dataframe.dropDuplicates(subset=fields)
    return df_unique

In [19]:
## filter future dates
def filter_future_events(dataframe):
    df_ts = dataframe.withColumn('event_time', upper(trim(col('event_time'))))
    #df_ts = df_ts.withColumn('event_time', to_timestamp(col('event_time'), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))
    df_ts = df_ts.filter(col('event_time') <= current_timestamp())
    return df_ts

In [20]:
## handles null value for the provided columns
## user_id rows, null product_id, quantity rows for event_type purchase, add_to_cart and checkout
def handle_nulls(dataframe):
    # checking null counts accross rows
    dataframe.withColumns({c: sum(col(c).isNull().cast('int')) for c in dataframe_clickstream.columns}).show()
    # replaces with 'unknown' value for columns - category, country, device
    df = dataframe.fillna('unknown', subset=['category', 'country', 'device'])
    # dropping for user_id, event_time, session_id
    df = df.dropna(how='any', subset=['user_id', 'event_time', 'session_id'])
    ## handle NULLs, negatives and 0 in quantity as per event_type
    df = df.filter(~((col('event_type').isin('add_to_cart', 'purchase', 'checkout')) & (col('quantity') <= 0) & (col('quantity').isNull())))
    ## handling null product per event_type
    df = df.filter(~((col('event_type').isin('add_to_cart', 'purchase', 'checkout')) & (col('product_id').isNull())))
    df.withColumns({c: sum(col(c).isNull().cast('int')) for c in dataframe_clickstream.columns}).show()
    # filters negative priced products
    df = df.filter(~((col('price').try_cast('double'))<=0))
    return df

In [21]:
## converts the schema type of all string schema to specfic types
def type_conversion(dataframe):
    schema1 = StructType([
    StructField('category', StringType(), nullable=True),
    StructField('country', StringType(), nullable=True),
    StructField('device', StringType(), nullable=True),
    StructField('event_id', StringType(), nullable=False),
    StructField('event_time', TimestampType(), nullable=False),
    StructField('event_type', StringType(), nullable=True),
    StructField('price', DoubleType(), nullable=True),
    StructField('product_id', StringType(), nullable=True),
    StructField('quantity', IntegerType(), nullable=True),
    StructField('session_id', StringType(), nullable=False),
    StructField('user_id', StringType(), nullable=False),
    StructField('ingestion_ts', TimestampType(), nullable=True),
    StructField('ip_file_name', StringType(), nullable=True),
    ])

    ## nullable = False fields: user_id, event_id, event_time, session_id
    ## this is done to ensure that no nulls value remain in these columns before converting them to nullable=False property
    ## even if there are no nulls, use coalesce anyways
    df = dataframe.withColumns({'user_id': coalesce('user_id', lit('unkonwn')),
                 'event_id': coalesce('event_id', lit('unkonwn')),
                  'event_time': coalesce('event_time', lit('unkonwn')),
                     'session_id': coalesce('session_id', lit('unknown'))
                 })
    df = df.withColumn('event_time', to_timestamp(col('event_time'), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))

    try:
        df = df.to(schema=schema1)
        return df
    except Exception as e:
        print(f'error: {e}')
        

In [22]:

dataframe_clickstream.printSchema()

root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [23]:
df = trim_everything_lower(dataframe_clickstream)
df = ensure_unique(df)
df = handle_nulls(dataframe_clickstream)
df = filter_future_events(df)
#df.printSchema()
df = type_conversion(df)
#df.printSchema()

+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|      10|      9|    13|       0|         0|         0|    0|        13|     888|         0|      3|           0|           0|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+

+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+--

In [24]:
df.printSchema()

root
 |-- category: string (nullable = false)
 |-- country: string (nullable = false)
 |-- device: string (nullable = false)
 |-- event_id: string (nullable = false)
 |-- event_time: timestamp (nullable = false)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- session_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [158]:
## checking nulls >> no nulls in user dataframe
dataframe_user.withColumns({c: sum(col(c).isNull().cast('int')) for c in dataframe_user.columns}).show()
dataframe_user.show(3, truncate= False)
dataframe_user.count()

+-----------+-------+-----+------------+-----------+----------+-------+------------+------------+
|change_type|country|email|loyalty_tier|signup_date|updated_at|user_id|ingestion_ts|ip_file_name|
+-----------+-------+-----+------------+-----------+----------+-------+------------+------------+
|          0|      0|    0|           0|          0|         0|      0|           0|           0|
+-----------+-------+-----+------------+-----------+----------+-------+------------+------------+

+-----------+-------+--------------------+------------+-----------+--------------------------------+--------+--------------------------+--------------------------------------------------------------------------------------------+
|change_type|country|email               |loyalty_tier|signup_date|updated_at                      |user_id |ingestion_ts              |ip_file_name                                                                                |
+-----------+-------+--------------------+-------

437

In [97]:
df_user = trim_everything_lower(dataframe_user)
print(df_user.count())

437


In [98]:
df_user.printSchema()

root
 |-- change_type: string (nullable = true)
 |-- country: string (nullable = true)
 |-- email: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: string (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [174]:
df_user.select('effective_from').distinct().show(10, truncate=False)

+--------------+
|effective_from|
+--------------+
|2026-01-14    |
|2026-02-22    |
|2025-12-03    |
|2026-01-20    |
|2026-06-20    |
|2025-10-27    |
|2026-08-06    |
|2025-09-23    |
|2026-06-27    |
|2026-04-11    |
+--------------+
only showing top 10 rows


In [203]:
def create_user_df(dataframe):
    df_user = trim_everything_lower(dataframe)
    
    # defining window spec
    window_spec_latest_user = Window.partitionBy('user_id').orderBy(col('updated_at').desc())
    ## using this window spec with rank function
    df_latest_user = df_user.withColumn('ranks', rank().over(window_spec_latest_user))

    ## adding is_current flag, so latest record show True and old ones false
    df_latest_user = df_latest_user.withColumn('is_current', when(col('ranks') == 1, True).otherwise(False))

    df_latest_user = df_latest_user.withColumnRenamed('signup_date', 'effective_from').withColumnRenamed('updated_at', 'effective_to')
    
    df_latest_user = df_latest_user.select('user_id', 'country', 'effective_from', 'effective_to', 'is_current')
    ## type conversion
    ## defining schema
    user_schema = StructType([
        StructField('user_id', StringType(), False),
        StructField('country', StringType(), True),
        StructField('effective_from', DateType(), False),
        StructField('effective_to', TimestampType(), True),
        StructField('is_current', BooleanType(), True)
    ])
    
    ## coalescing not-null fields for precaution without date/datetime casting
    df_latest_user = df_latest_user.withColumns({
        'user_id': coalesce(col('user_id'), lit('unknown')),
        'effective_from': coalesce(upper(col('effective_from')), lit('1970-01-01')),
        'effective_to': coalesce(upper(col('effective_to')), lit("9999-12-31 23:59:00"))
    })

    ##changing type for date columns
    df_latest_user = df_latest_user.withColumns({
        'effective_from': to_date('effective_from', 'yyyy-MM-dd'),
        'effective_to': to_timestamp('effective_to', "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX")
    })

    df_latest_user = df_latest_user.to(user_schema)
    return df_latest_user

In [204]:
dataframe_user.select('updated_at').show(2,truncate=False)

+--------------------------------+
|updated_at                      |
+--------------------------------+
|2025-09-06T19:10:08.537059+00:00|
|2025-09-06T19:10:08.537059+00:00|
+--------------------------------+
only showing top 2 rows


In [205]:
df_user = create_user_df(dataframe_user)
df_user.printSchema()
df_user.show(5, truncate=False)

root
 |-- user_id: string (nullable = false)
 |-- country: string (nullable = true)
 |-- effective_from: date (nullable = false)
 |-- effective_to: timestamp (nullable = false)
 |-- is_current: boolean (nullable = false)

+-------+-----------+--------------+--------------------------+----------+
|user_id|country    |effective_from|effective_to              |is_current|
+-------+-----------+--------------+--------------------------+----------+
|user_1 |netherlands|2025-12-28    |2026-10-03 04:29:51.537059|true      |
|user_1 |germany    |2025-12-28    |2026-09-23 11:55:33.537059|false     |
|user_1 |germany    |2025-12-28    |2025-12-29 00:55:08.537059|false     |
|user_10|netherlands|2025-10-26    |2026-09-22 03:53:47.537059|true      |
|user_10|germany    |2025-10-26    |2026-09-11 00:35:18.537059|false     |
+-------+-----------+--------------+--------------------------+----------+
only showing top 5 rows


In [120]:
#df_user_new = create_user_df(df_user)
#df_user_new.show(10)

In [91]:
df_latest_user.select('user_id', 'updated_at', 'is_current').orderBy('user_id').distinct().show(truncate=False)

+--------+--------------------------------+----------+
|user_id |updated_at                      |is_current|
+--------+--------------------------------+----------+
|user_1  |2026-10-02t22:44:51.537059+00:00|true      |
|user_1  |2026-09-23t06:10:33.537059+00:00|false     |
|user_1  |2025-12-28t19:10:08.537059+00:00|false     |
|user_10 |2026-09-21t22:08:47.537059+00:00|true      |
|user_10 |2026-09-10t18:50:18.537059+00:00|false     |
|user_10 |2025-10-26t19:10:08.537059+00:00|false     |
|user_100|2026-09-29t23:40:34.537059+00:00|true      |
|user_100|2026-09-29t08:40:53.537059+00:00|false     |
|user_100|2026-09-27t13:10:53.537059+00:00|false     |
|user_100|2026-07-05t19:10:08.537059+00:00|false     |
|user_101|2026-03-20t19:10:08.537059+00:00|true      |
|user_102|2026-09-26t20:50:44.537059+00:00|true      |
|user_102|2026-09-22t02:48:23.537059+00:00|false     |
|user_102|2026-09-16t17:49:02.537059+00:00|false     |
|user_102|2025-11-01t19:10:08.537059+00:00|false     |
|user_103|

In [66]:
df_latest_user_only = df_latest_user.filter(col('rank') == 1)
df_latest_user_only.count()

200

In [73]:
df_latest_user.groupBy('user_id').count().show()

+--------+-----+
| user_id|count|
+--------+-----+
| user_58|    1|
|user_178|    1|
|user_151|    1|
| user_94|    1|
|user_166|    4|
| user_73|    1|
| user_85|    1|
| user_14|    4|
| user_56|    4|
| user_68|    1|
| user_22|    1|
| user_97|    1|
| user_86|    4|
|user_115|    3|
|user_157|    4|
| user_65|    1|
|user_172|    3|
| user_47|    4|
| user_66|    1|
|user_120|    3|
+--------+-----+
only showing top 20 rows


In [81]:
df_latest_user.select('user_id').distinct().count()

200